# AIC211 · Topic 2 · TV3 (Trí, SE211087) — E2b: đối chứng hình dạng URL

**Câu hỏi:** model có học luật “ngắn, không đường dẫn = độc” không?
**Ba phép đo:** (1) tỉ lệ trang chủ theo lớp ở A-train; (2) ở A-test, FPR/recall tách theo trang chủ và có path; (3) ở B, chỉ giữ trang chủ của cả hai lớp rồi đo PR-AUC.

| Ô | Làm gì | Thời gian |
|---|---|---|
| 1–2 | Nạp thư viện, dữ liệu, 3 cấu hình | 2–3 phút |
| 3 | Phép đo 1 | vài giây |
| 4 | Chấm điểm A-test và B cho 3 cấu hình | 3–8 phút |
| 5 | Phép đo 2, 3 | vài giây |
| 6 | Lưu bảng, hình; ghi `results.csv` (chỉ khi `LOG = True`) | vài giây |
| 7 | Thử nghiệm dấu “/” cuối (bằng chứng chính) | dưới 1 phút |
| 8 | Vẽ hình thử nghiệm dấu “/” (`figures/E2b_slash_test.png`) | vài giây |

**Lưu ý:** E2b đã được ghi vào `results.csv` ngày 24–25/10. Giữ `LOG = False` ở Ô 6; chỉ cần chạy lại Ô 1–5 (để có dữ liệu trong bộ nhớ) rồi Ô 8 nếu muốn vẽ lại hình.

**Không sửa file nào trong `src/` hay `models/`.** Notebook này chỉ đọc, và chỉ ghi 3 thứ: `results/E2b_shape.csv`, `figures/E2b_shape.png`, các dòng `exp = E2b_shape` trong `results.csv`.

In [ ]:
# Ô 1 — Cài thư viện + nối Google Drive
import os, sys, importlib
from google.colab import drive
drive.mount('/content/drive')
!pip -q install tldextract rapidfuzz shap xgboost
ROOT = '/content/drive/MyDrive/AIC211_URL'
sys.path.insert(0, f'{ROOT}/src')
print('ROOT =', ROOT); print(sorted(os.listdir(ROOT)))

In [ ]:
# Ô 2 — Nạp code, dữ liệu, 3 cấu hình (không phòng thủ RF_F1, LR_F2, D1+D2+D3)
import common; importlib.reload(common)
from common import *
import json, joblib, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score
import e3, e4, defenses, d2
for m in (e3, defenses, d2, e4): importlib.reload(m)
from e3 import make_score_fn, FEATURIZE
from e4 import load_configs

A = pd.read_parquet(f'{ROOT}/data/processed/A.parquet')
B = pd.read_parquet(f'{ROOT}/data/processed/B.parquet')
THR = json.load(open(f'{ROOT}/models/thresholds.json'))
INFO = json.load(open(f'{ROOT}/models/model_info.json'))

CONF = {}
for n in ['RF_F1', 'LR_F2']:
    m = joblib.load(f'{ROOT}/models/{n}.joblib')
    CONF[n] = {'score': make_score_fn(m, FEATURIZE[INFO[n]['features']]), 'thr': THR[n]}
CFG = load_configs(ROOT)
print('Cấu hình trong e4.load_configs:', list(CFG))
CONF['RF_D1+D2+D3'] = {'score': CFG['D1+D2+D3']['score'], 'thr': CFG['D1+D2+D3']['thr']}
for n, v in CONF.items(): print(f'{n:12s} ngưỡng = {v["thr"]:.4f}')

# 'Trang chủ' = phần sau host dài <= 1 ký tự (rỗng hoặc chỉ '/')
for df in (A, B): df['home'] = df['rest'].fillna('').str.len() <= 1
print('A:', A.shape, '| B:', B.shape)

In [ ]:
# Ô 3 — Phép đo 1: tỉ lệ trang chủ theo lớp và nguồn (luật có sẵn trong dữ liệu học không?)
def home_share(df, name):
    g = df.groupby('label')['home'].agg(['mean', 'size'])
    return {'tập': name,
            'n sạch': int(g.loc[0, 'size']), '% trang chủ ở URL sạch': round(100 * g.loc[0, 'mean'], 1),
            'n độc': int(g.loc[1, 'size']), '% trang chủ ở URL độc': round(100 * g.loc[1, 'mean'], 1)}
tr = (A.split == 'train').values; te = (A.split == 'test').values
P1 = pd.DataFrame([home_share(A[tr], 'A-train'), home_share(A[te], 'A-test'), home_share(B, 'B')])
print(P1.to_string(index=False))
print('\nĐọc: nếu ở A-train URL độc là trang chủ nhiều hơn hẳn URL sạch, luật "không path = độc" có sẵn trong dữ liệu học.')

In [ ]:
# Ô 4 — Chấm điểm một lần cho A-test và B (3 cấu hình). B có ~214k URL nên mất vài phút.
SC = {}
t0 = time.time()
for src, df in [('A-test', A[te]), ('B', B)]:
    urls = ['http://' + u for u in df['url_norm']]   # cùng dạng đầu vào như Ô 5 notebook TV1 tuần 7
    for n, v in CONF.items():
        SC[(n, src)] = np.asarray(v['score'](urls), dtype=float)
        print(f'{n:12s} {src:6s} xong ({time.time() - t0:.0f}s)')

In [ ]:
# Ô 5 — Phép đo 2 (A-test) và 3 (B chỉ trang chủ): FPR, recall, PR-AUC theo nhóm hình dạng
def metrics(y, s, thr):
    y = np.asarray(y); pred = s > thr
    nb, nm = int((y == 0).sum()), int((y == 1).sum())
    out = {'n_benign': nb, 'n_malicious': nm,
           'fpr': float(pred[y == 0].mean()) if nb else np.nan,
           'recall': float(pred[y == 1].mean()) if nm else np.nan,
           'pr_auc': float(average_precision_score(y, s)) if nb and nm else np.nan}
    return out

rows = []
for src, df in [('A-test', A[te]), ('B', B)]:
    y, home = df['label'].values, df['home'].values
    for n, v in CONF.items():
        s = SC[(n, src)]
        for grp, mask in [('all', np.ones(len(df), bool)), ('home', home), ('path', ~home)]:
            if mask.sum() == 0: continue
            rows.append({'config': n, 'source': src, 'group': grp, **metrics(y[mask], s[mask], v['thr'])})
E2b = pd.DataFrame(rows)
show = E2b.copy()
for col in ['fpr', 'recall', 'pr_auc']: show[col] = (100 * show[col]).round(1) if col != 'pr_auc' else show[col].round(3)
print('FPR, recall tính bằng %; PR-AUC từ 0 đến 1. NaN = nhóm đó không có đủ hai lớp.')
print(show.to_string(index=False))

In [ ]:
# Ô 6 — Lưu bảng + hình; ghi results.csv khi LOG = True (nhắn nhóm chat trước khi bật)
LOG = False
E2b.to_csv(f'{ROOT}/results/E2b_shape.csv', index=False)
P1.to_csv(f'{ROOT}/results/E2b_home_share.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
a = E2b[(E2b.source == 'A-test') & (E2b.group != 'all')]
piv = a.pivot(index='config', columns='group', values='fpr') * 100
piv[['home', 'path']].plot.bar(ax=axes[0], color=['#2f6db5', '#a9a9a9'], rot=0)
axes[0].set_title('A-test: FPR trên URL sạch, theo hình dạng'); axes[0].set_ylabel('FPR (%)'); axes[0].set_xlabel('')
b = E2b[(E2b.source == 'B') & (E2b.group.isin(['all', 'home']))]
piv2 = b.pivot(index='config', columns='group', values='pr_auc')
piv2.plot.bar(ax=axes[1], color=['#a9a9a9', '#2f6db5'], rot=0)
axes[1].set_title('B: PR-AUC toàn bộ vs chỉ trang chủ'); axes[1].set_ylim(0, 1); axes[1].set_xlabel('')
for ax in axes:
    for s in ['top', 'right']: ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig(f'{ROOT}/figures/E2b_shape.png', dpi=200); plt.show()

if LOG:
    for r in E2b.itertuples():
        met = {'fpr': r.fpr, 'recall': r.recall, 'pr_auc': r.pr_auc, 'n_benign': r.n_benign, 'n_malicious': r.n_malicious}
        log_result(ROOT, 'E2b_shape', r.config, f'{r.source}|{r.group}', met)
    print('Đã ghi', len(E2b), 'dòng exp = E2b_shape vào results.csv')
else:
    print('Chưa ghi results.csv (LOG = False). Đã lưu E2b_shape.csv, E2b_home_share.csv, E2b_shape.png.')

In [ ]:
# Ô 7 — Thử nghiệm dấu "/" cuối: chỉ thêm hoặc bỏ MỘT ký tự ở trang chủ, xem quyết định đổi thế nào
SETS = {'B sạch, trang chủ (không "/")':   (B[B.home & (B.label == 0)].sample(3000, random_state=0), 'add'),
        'A-test sạch, trang chủ (có "/")': (A[te & A.home.values & (A.label == 0).values], 'drop'),
        'A-test độc, trang chủ (không "/")': (A[te & A.home.values & (A.label == 1).values], 'add')}
rows7 = []
for name, (df, op) in SETS.items():
    orig = ['http://' + u for u in df['url_norm']]
    mod = ['http://' + (u.rstrip('/') + '/' if op == 'add' else u.rstrip('/')) for u in df['url_norm']]
    for n, v in CONF.items():
        before = float((np.asarray(v['score'](orig)) > v['thr']).mean())
        after = float((np.asarray(v['score'](mod)) > v['thr']).mean())
        rows7.append({'group': name, 'op': 'add_slash' if op == 'add' else 'drop_slash', 'config': n,
                      'blocked_before': before, 'blocked_after': after})
SLASH = pd.DataFrame(rows7)
SLASH.to_csv(f'{ROOT}/results/E2b_slash_test.csv', index=False)
show7 = SLASH.copy(); show7[['blocked_before', 'blocked_after']] = (100 * show7[['blocked_before', 'blocked_after']]).round(1)
print('% URL bị chặn trước / sau khi thêm hoặc bỏ đúng một dấu "/":'); print(show7.to_string(index=False))
if LOG:
    for r in SLASH.itertuples():
        log_result(ROOT, 'E2b_slash', r.config, r.group, {'blocked_before': r.blocked_before, 'blocked_after': r.blocked_after}, ops=r.op)
    print('Đã ghi', len(SLASH), 'dòng exp = E2b_slash')

In [ ]:
# Ô 8 — Vẽ hình thử nghiệm dấu "/" (hình chính cho báo cáo, slide) từ E2b_slash_test.csv — chỉ đọc, không ghi results.csv
S = pd.read_csv(f'{ROOT}/results/E2b_slash_test.csv')
S[['blocked_before', 'blocked_after']] *= 100
fig, ax = plt.subplots(figsize=(10, 6.2)); y, yt, yl = 0, [], []
for grp, d in S.groupby('group', sort=False):
    ax.text(-2, y + 0.5, grp, fontweight='bold', va='bottom')
    for r in d.itertuples():
        y -= 1; c = '#a9a9a9' if r.config == 'LR_F2' else '#2f6db5'
        if abs(r.blocked_after - r.blocked_before) > 0.5:
            ax.annotate('', xy=(r.blocked_after, y), xytext=(r.blocked_before, y), arrowprops=dict(arrowstyle='-|>', color=c, lw=2))
        ax.plot(r.blocked_before, y, 'o', mfc='white', mec=c, mew=2, ms=8, zorder=3)
        ax.plot(r.blocked_after, y, 'o', color=c, ms=8, zorder=3)
        ax.text(r.blocked_before, y + 0.3, f'{r.blocked_before:.1f}%', ha='center', fontsize=8, color='#666666')
        if abs(r.blocked_after - r.blocked_before) > 0.5:
            ax.text(r.blocked_after, y + 0.3, f'{r.blocked_after:.1f}%', ha='center', fontsize=8.5, fontweight='bold')
        yt.append(y); yl.append(r.config)
    y -= 1.4
ax.set_yticks(yt); ax.set_yticklabels(yl); ax.set_xlim(-3, 108); ax.set_ylim(y + 0.6, 1.3)
ax.set_xlabel('% bị chặn  (○ trước, ● sau khi thêm/bỏ đúng một dấu "/")')
for s in ['top', 'right', 'left']: ax.spines[s].set_visible(False)
ax.grid(axis='x', alpha=.25)
ax.set_title('Thêm hoặc bỏ đúng MỘT dấu "/" lật ngược quyết định của RF', loc='left', fontweight='bold')
plt.tight_layout(); plt.savefig(f'{ROOT}/figures/E2b_slash_test.png', dpi=200); plt.show()
print('Đã lưu figures/E2b_slash_test.png')

## Cách đọc kết quả (Claude đã chạy thử trên bản sao Drive ngày 24/10 – số dự kiến)

- **Ô 3:** ở A-train, 21,1% URL độc là trang chủ, so với 11,0% URL sạch. Nhưng điểm mấu chốt nằm ở dấu “/”: 99,6% trang chủ **sạch** của A kết thúc bằng “/”, còn 92,2% trang chủ **độc** thì không. 100% URL sạch của B không có “/”.
- **Ô 5:** trên A-test, RF_F1 bắt 92,6% URL độc là trang chủ nhưng chỉ 24,9% URL độc có path. Ở B, khi chỉ giữ trang chủ của cả hai lớp, PR-AUC của RF_F1 là 0,214 – **thấp hơn mức đoán bừa** (0,307), tức thứ hạng bị đảo. LR_F2 đạt 0,555.
- **Ô 7 (bằng chứng chính):** thêm “/” vào trang chủ sạch của B → RF_F1 chặn nhầm từ 100% xuống 0%. Bỏ “/” ở trang chủ sạch của A-test → từ 0,4% lên 99,6%. Thêm “/” vào trang chủ độc của A-test → recall từ 92,6% xuống 0,2%. D1+D2+D3 y hệt. LR_F2 không đổi, vì F2 dùng cụm 3–5 ký tự nên một dấu “/” đứng một mình không tạo ra feature nào.
- **Kết luận:** RF học luật “có dấu / cuối = sạch”. Đây là artifact của cách thu thập nguồn A, giải thích cùng lúc phát hiện 1 (chặn nhầm 100% ở B) và phát hiện 3 (T1 lọt qua D1 nhờ dấu “/”).

Xong khi: có `results/E2b_shape.csv`, `E2b_home_share.csv`, `E2b_slash_test.csv`, `figures/E2b_shape.png`, và (sau khi báo nhóm, `LOG = True`) các dòng `E2b_shape`, `E2b_slash` trong `results.csv`.